In [70]:
# Download dataset directly using kagglehub (no kaggle.json or manual unzipping needed)
!pip install -q kagglehub
import kagglehub

path = kagglehub.dataset_download("jangedoo/utkface-new")
print("Dataset downloaded to:", path)

100%|██████████| 331M/331M [00:08<00:00, 38.8MB/s] 

Extracting files...


Dataset downloaded to: /root/.cache/kagglehub/datasets/jangedoo/utkface-new/versions/1


In [71]:
import os
import numpy as np
import pandas as pd
import tensorflow as tf
import keras
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Conv2D, MaxPooling2D, Flatten, Dense
from tensorflow.keras.preprocessing.image import ImageDataGenerator

In [72]:
# Locate the UTKFace images directory inside the downloaded dataset
folder_path = os.path.join(path, "utkface_aligned_cropped", "UTKFace")

# Fallback check in case the directory is structured directly as UTKFace or flat
if not os.path.exists(folder_path):
    folder_path = os.path.join(path, "UTKFace")
if not os.path.exists(folder_path):
    for root, dirs, files in os.walk(path):
        if any(f.endswith(".chip.jpg") or (f.endswith(".jpg") and "_" in f) for f in files):
            folder_path = root
            break

print("Image folder path:", folder_path)

Image folder path: /root/.cache/kagglehub/datasets/jangedoo/utkface-new/versions/1/utkface_aligned_cropped/UTKFace


In [73]:
age = []
gender = []
img_path = []
for file in os.listdir(folder_path):
    try:
        age.append(int(file.split("_")[0]))
        gender.append(int(file.split("_")[1]))
        img_path.append(file)
    except:
        continue

In [74]:
len(age)

23708

In [75]:
df = pd.DataFrame({'age':age,'gender':gender,'img':img_path})

In [76]:
df.shape

(23708, 3)

In [77]:
df.head()

,age,gender,img
0,35,1,35_1_0_20170117191417058.jpg.chip.jpg
1,42,1,42_1_0_20170110154309957.jpg.chip.jpg
2,2,0,2_0_2_20161219163415222.jpg.chip.jpg
3,61,0,61_0_0_20170105173647189.jpg.chip.jpg
4,29,1,29_1_1_20170113001147795.jpg.chip.jpg


In [78]:
train_df = df.sample(frac=1,random_state=0).iloc[:20000]
test_df = df.sample(frac=1,random_state=0).iloc[20000:]

In [79]:
train_df.shape

(20000, 3)

In [80]:
test_df.shape

(3708, 3)

In [81]:
train_datagen = ImageDataGenerator(rescale=1./255,
                                   rotation_range=30,
                                   width_shift_range=0.2,
                                   height_shift_range=0.2,
                                   shear_range=0.2,
                                   zoom_range=0.2,
                                   horizontal_flip=True)

test_datagen = ImageDataGenerator(rescale=1./255)

In [82]:
train_generator = train_datagen.flow_from_dataframe(train_df,
                                                    directory=folder_path,
                                                    x_col='img',
                                                    y_col=['age','gender'],
                                                    target_size=(200,200),
                                                    class_mode='multi_output')

test_generator = test_datagen.flow_from_dataframe(test_df,
                                                  directory=folder_path,
                                                  x_col='img',
                                                  y_col=['age','gender'],
                                                  target_size=(200,200),
                                                  class_mode='multi_output')

# Helper generator yielding targets as a tuple instead of a list
# Resolves Keras 3 / TensorFlow TypeError: output_signature must contain TypeSpec
def multi_output_generator(generator):
    for x, y in generator:
        yield x, (y[0], y[1])


Found 20000 validated image filenames.
Found 3708 validated image filenames.


In [83]:
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Conv2D, MaxPooling2D, Flatten, Dense
from keras.applications.resnet50 import ResNet50
from keras.layers import *
from keras.models import Model

In [84]:
resnet = ResNet50(include_top=False, input_shape=(200,200,3))

In [85]:
resnet = ResNet50(include_top=False, input_shape=(200,200,3))

resnet.trainable=False

output = resnet.layers[-1].output

flatten = Flatten()(output)

dense1 = Dense(512, activation='relu')(flatten)
dense2 = Dense(512,activation='relu')(flatten)

dense3 = Dense(512,activation='relu')(dense1)
dense4 = Dense(512,activation='relu')(dense2)

output1 = Dense(1,activation='linear',name='age')(dense3)
output2 = Dense(1,activation='sigmoid',name='gender')(dense4)

In [86]:
model = Model(inputs=resnet.input,outputs=[output1,output2])

In [87]:
model.compile(optimizer='adam', loss={'age': 'mae', 'gender': 'binary_crossentropy'}, metrics={'age': 'mae', 'gender': 'accuracy'},loss_weights={'age':1,'gender':99})

In [88]:
model.fit(
    multi_output_generator(train_generator),
    steps_per_epoch=len(train_generator),
    epochs=10,
    validation_data=multi_output_generator(test_generator),
    validation_steps=len(test_generator)
)


Epoch 1/10
625/625 ━━━━━━━━━━━━━━━━━━━━ 294s 429ms/step - age_loss: 15.5241 - age_mae: 15.5241 - gender_accuracy: 0.5149 - gender_loss: 0.8342 - loss: 98.1138 - val_age_loss: 14.7460 - val_age_mae: 14.7464 - val_gender_accuracy: 0.5372 - val_gender_loss: 0.6908 - val_loss: 83.1354
Epoch 2/10
625/625 ━━━━━━━━━━━━━━━━━━━━ 247s 396ms/step - age_loss: 14.9936 - age_mae: 14.9936 - gender_accuracy: 0.5192 - gender_loss: 0.6937 - loss: 83.6661 - val_age_loss: 14.6467 - val_age_mae: 14.6467 - val_gender_accuracy: 0.5407 - val_gender_loss: 0.6908 - val_loss: 83.0390
Epoch 3/10
625/625 ━━━━━━━━━━━━━━━━━━━━ 247s 395ms/step - age_loss: 14.9243 - age_mae: 14.9243 - gender_accuracy: 0.5199 - gender_loss: 0.6936 - loss: 83.5930 - val_age_loss: 14.5201 - val_age_mae: 14.5201 - val_gender_accuracy: 0.5391 - val_gender_loss: 0.6915 - val_loss: 82.9783
Epoch 4/10
625/625 ━━━━━━━━━━━━━━━━━━━━ 253s 405ms/step - age_loss: 14.7486 - age_mae: 14.7486 - gender_accuracy: 0.5202 - gender_loss: 0.6926 - loss: 83.